# Two-Phase Bayesian Optimization — NYX & Miranda — **SPERR base** (1×4)

SPERR-base twin of `nyx_miranda.ipynb`: identical Phase-1/Phase-2 code, only the base
compressor differs. SPERR runs in the pipeline's `--psnr` (MSE-optimal) mode and its
target PSNR is bisected so the SPERR stream lands at the SAME compression ratio as the
SZ3 base of the SZ3 figure (NYX rel 1e-5, Miranda rel 7e-3), so the two figures are
comparable operating points. Output: `SPERR_NYX_Miranda_1x4.pdf`.

Runs the joint **LR × slice-direction** two-phase BO pipeline on **both** datasets
and renders the combined figure:

|          | **Phase 1** (proxy BO) | **Phase 2** (full-res validation) |
|----------|------------------------|-----------------------------------|
| **NYX**  | top-left               | top-right                         |
| **Miranda**| bottom-left            | bottom-right                      |

Budgets: **NYX = 10 s**, **Miranda = 80 s**.  For each: **Phase 1 = 10 %** of the
budget, **Phase 2 = the remaining 90 %**, **10 trials** each.

NYX is a multifield 512³ volume (baryon_density target + 5 aux fields); Miranda is a
single-field 1024³ volume (float32), no aux conditioning.32 so the 2-D slices are square 256×256 (SZ3 rel-err = 3e-2).

The shared `run_two_phase(...)` function runs the LR × slice-direction BO and returns
everything the combined plot needs.

In [1]:
import random, sys, os, copy, time
from pathlib import Path

import numpy as np
import torch
import matplotlib.pyplot as plt
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..", "base_script")))  # <repo>/base_script (run from this folder)
from local_paths import P   # machine-specific paths: env var > <repo>/local_paths.env > placeholder

from config_io import load_multifield_from_disk
# Force-reload edited modules so re-running picks up bg_stage.py changes WITHOUT
# restarting the kernel (Jupyter caches modules in sys.modules).
import importlib
import bg_stage
importlib.reload(bg_stage)

from experiment import build_bg_only_cfg
from bg_stage import run_bg_inference, train_bg_only, unwrap_bg_model
from bg_shard import pick_bg_h_under_budget

pysz_dir = P("ADAMIT_PYSZ")
if pysz_dir not in sys.path:
    sys.path.append(pysz_dir)
from pysz import SZ

def set_seed(seed=42):
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

# cuda:0 after CUDA_VISIBLE_DEVICES filtering -- hard-coding cuda:1 breaks whenever the
# notebook is pinned to a single GPU by UUID (only one ordinal is then visible).
# BO_DEVICE overrides if a specific ordinal is really wanted.
device = torch.device(os.environ.get("BO_DEVICE", "cuda:0") if torch.cuda.is_available() else "cpu")
print(f"Device: {device} | GPUs: {torch.cuda.device_count()}")

~/miniconda3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cuda:0 | GPUs: 2


In [2]:
# ── Speed vs reproducibility ─────────────────────────────────────────────────
# Steady-state cost of one Miranda 1024^3 epoch (1024 steps, patch 1024, bg_h=61) on
# an RTX PRO 6000, measured over 3 epochs with the first discarded:
#
#   bf16 + deterministic cuDNN     28 s   <- current setting
#   bf16 + cuDNN benchmark         28 s   (autotune buys nothing; its first epoch is
#                                          44 s vs 30 s while it searches algorithms)
#   fp32 + deterministic cuDNN     49 s
#   fp32 + STRICT_REPRO           130 s
#
# Two counterintuitive results worth keeping. cuDNN autotune does not help here, so
# deterministic cuDNN is left on: it is free and removes one source of drift. And the
# expensive flag is STRICT_REPRO, not precision -- the model uses
# nn.Upsample(mode='bilinear'), whose backward accumulates with atomicAdd, and forcing
# a deterministic kernel for it is the whole 49 -> 130 s. cuDNN flags alone do NOT give
# bit-exactness (PSNR still drifts ~0.003 dB run to run), so it is all-or-nothing.
#
# USE_AMP is a quality knob, independent of the two below. bf16 costs 0.09 dB on
# Miranda but 0.91 dB on NYX (sec_3_4_bf16_storage/bf_16.ipynb, amp the only variable): NYX sits
# near 125 dB where bf16 mantissa noise bites. Set it False if NYX fidelity matters
# more than the ~1.7x speedup.
#
# Bit-exactness also needs the step count pinned -- a wall-clock budget lets the machine
# decide how much training happens (two timed runs measured 961 vs 1002 steps). Record
# history["total_steps"] from a timed run and replay it through cfg.bg_max_steps.
# REPRO_EPOCHS is the blunter option: it swaps the budget for a fixed epoch count and
# therefore abandons the paper's fixed-time protocol.
USE_AMP       = True     # bf16 -> ~28 s/epoch; False -> fp32, ~49 s, +0.91 dB on NYX
DETERMINISTIC = True     # deterministic cuDNN (free, and faster than autotune here)
STRICT_REPRO  = False    # + deterministic algorithms & no TF32 -> bit-exact, 2.7x slower
REPRO_EPOCHS  = None     # int -> fixed epochs instead of the wall-clock budget

if STRICT_REPRO:
    os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
    torch.use_deterministic_algorithms(True, warn_only=True)
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False
else:
    torch.use_deterministic_algorithms(False, warn_only=True)
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True


# ── Slice-direction permutations ──────────────────────────────────────────────
# Data layout: (Z, Y, X). Permute so the target slice axis becomes axis-0 (depth).
DIRECTIONS = ["Z", "Y", "X"]
_FWD = {"Z": (0, 1, 2), "Y": (1, 0, 2), "X": (2, 0, 1)}
_INV = {"Z": (0, 1, 2), "Y": (1, 0, 2), "X": (1, 2, 0)}
_DIR_AXIS = {"Z": 0, "Y": 1, "X": 2}


# np.transpose() alone is a free view; it was the ascontiguousarray() wrapped around
# it that materialised a full extra copy (4 GB per field on Miranda 1024^3, x2 fields
# for permute + another 4 GB for unpermute -> a large part of the OOM kills).
#
# Whether dropping that copy is worth it depends entirely on the axis, because the two
# permutations have very different stride patterns. Measured end-to-end on the real
# model (run_bg_inference, N=512; output verified bit-identical both ways):
#
#   permute   Y (1,0,2): view +0.02 s/epoch  <- free; drop the copy
#   permute   X (2,0,1): view +2.44 s/epoch  <- one-off copy (1.1 s) is far cheaper
#   unpermute Y/X      : consumed once by a whole-array PSNR reduction, view is a wash
#
# So X keeps its materialised copy on the read-heavy permute path, everything else
# stays a view. Z needs no permutation at all.
_PERM_NEEDS_COPY = {"Y": False, "X": True}


def permute_fields(fields, direction):
    axes = _FWD[direction]
    if axes == (0, 1, 2):
        return fields
    if _PERM_NEEDS_COPY[direction]:
        return [np.ascontiguousarray(np.transpose(f, axes)) for f in fields]
    return [np.transpose(f, axes) for f in fields]


def unpermute_field(field, direction):
    axes = _INV[direction]
    if axes == (0, 1, 2):
        return field
    return np.transpose(field, axes)


def build_cfg(Xs_in, Xps_in, max_train_time, bg_h, steps_per_epoch,
              lr=1e-4, epochs=200, log_prefix="", patch_size=None):
    if patch_size is None:
        patch_size = Xs_in[0].shape[2]
    if REPRO_EPOCHS is not None:
        # Fixed epoch count -> reproducible. Drops the wall-clock protocol.
        max_train_time, epochs = 1e9, int(REPRO_EPOCHS)
    cfg = build_bg_only_cfg(
        X_target=Xs_in[0], Xps=Xps_in,
        max_train_time=max_train_time, epochs=epochs,
        steps_per_epoch=steps_per_epoch, bg_h=bg_h,
        bg_batch=1, bg_patch_size=patch_size, lr=lr,
    )
    cfg.bg_sample_mode   = os.environ.get("BO_SAMPLE_MODE", "shuffled")   # paper protocol; "shuffled" = every slice once per epoch, random order
    cfg.bg_log_prefix    = log_prefix
    cfg.bg_arch          = "spatial"
    cfg.amp              = bool(USE_AMP)
    cfg.amp_dtype        = "bf16"
    cfg.bg_ddp           = False
    cfg.bg_data_parallel = False
    cfg.seed                   = 42
    cfg.bg_cudnn_deterministic = bool(DETERMINISTIC)
    cfg.bg_cudnn_benchmark     = not bool(DETERMINISTIC)
    return cfg


def psnr_from_arrays(target, recon):
    data_range = float(target.max() - target.min())
    if data_range <= 0:
        data_range = 1.0
    mse = float(np.mean((target - recon) ** 2))
    return 100.0 if mse <= 0 else 20.0 * np.log10(data_range) - 10.0 * np.log10(mse)


# Shared plot styling
dir_colors  = {"Z": "#1f77b4", "Y": "#ff7f0e", "X": "#2ca02c"}
dir_markers = {"Z": "o",       "Y": "s",        "X": "^"}

print("Helpers ready")

Helpers ready


In [3]:
BO_TAG = "sperr"   # base compressor tag: names bo_results/ pickles and the output PDF
# ── Paths, SZ engine (only to find the matching CR) and SPERR base ─────────────
import subprocess
# data locations come from local_paths (ADAMIT_NYX_DIR / ADAMIT_MIRANDA_FILE)
sz_lib_path = P("ADAMIT_SZ3_LIB")
sz_engine   = SZ(sz_lib_path)
SPERR_BIN   = P("ADAMIT_SPERR_BIN")
print("SZ engine loaded | sperr3d:", os.path.exists(SPERR_BIN))


def _sperr_env():
    """LD_LIBRARY_PATH extended to wherever libSPERR.so* lives."""
    import glob as _glob
    env = os.environ.copy(); root = os.path.dirname(SPERR_BIN); found = set()
    for _ in range(4):
        for hit in _glob.glob(os.path.join(root, "**", "libSPERR.so*"), recursive=True):
            found.add(os.path.dirname(hit))
        root = os.path.dirname(root)
        if not root or root == "/":
            break
    if found:
        env["LD_LIBRARY_PATH"] = ":".join(found) + ((":" + env["LD_LIBRARY_PATH"])
                                                     if env.get("LD_LIBRARY_PATH") else "")
    return env
_SPERR_ENV = _sperr_env()


def run_sperr_psnr(data_file, shape, target_psnr):
    """sperr3d in --psnr mode (what SPERR_fft.py uses). Returns (recon f32, nbytes)."""
    W, H, D = shape[2], shape[1], shape[0]
    tag = f"{os.getpid()}_{time.time_ns()}"
    bit = f"/tmp/sperr_{tag}.bit"; rec = f"/tmp/sperr_{tag}.dec.f32"
    r = subprocess.run([SPERR_BIN, "-c", "--ftype", "32", "--dims", str(W), str(H), str(D),
                        "--psnr", f"{float(target_psnr):.4f}", "--bitstream", bit, data_file],
                       capture_output=True, text=True, env=_SPERR_ENV)
    if not os.path.exists(bit):
        raise RuntimeError(f"SPERR compress failed: {r.stderr[:300]}")
    nbytes = os.path.getsize(bit)
    subprocess.run([SPERR_BIN, "-d", "--decomp_f", rec, bit], capture_output=True, text=True, env=_SPERR_ENV)
    recon = np.fromfile(rec, dtype=np.float32).reshape(shape)
    for f in (bit, rec):
        if os.path.exists(f):
            os.remove(f)
    return recon, int(nbytes)


def sperr_at_cr(data_file, vol, target_cr, lo=20.0, hi=200.0, iters=12, label="SPERR"):
    """Bisect --psnr so SPERR's CR ~= target_cr (CR falls as the target rises).
    Returns (recon, nbytes, cr, rel_err) with rel_err = max|err|/range (the pipeline's clamp)."""
    orig = int(vol.nbytes)
    for it in range(iters):
        mid = 0.5 * (lo + hi)
        recon, nb = run_sperr_psnr(data_file, vol.shape, mid)
        cr = orig / nb
        print(f"    [{label}] {it+1:2d}/{iters} psnr={mid:6.1f} -> CR {cr:7.1f} (target {target_cr:.1f})", flush=True)
        if cr > target_cr:
            lo = mid
        else:
            hi = mid
    recon, nb = run_sperr_psnr(data_file, vol.shape, 0.5 * (lo + hi))
    cr = orig / nb
    dr = float(vol.max() - vol.min()) or 1.0
    rel = float(np.abs(vol.astype(np.float32) - recon).max()) / dr
    print(f"    [{label}] picked CR {cr:.1f}  base PSNR {20*np.log10(dr)-10*np.log10(float(np.mean((vol.astype(np.float64)-recon)**2))):.2f} dB  rel(max|err|/range)={rel:.3e}")
    return recon, int(nb), float(cr), rel


# ── NYX: multifield 512^3 (baryon_density target + 5 aux fields) ──────────────
NYX_BASE = P("ADAMIT_NYX_DIR")
NYX_SHAPE = (512, 512, 512)
NYX_FIELD_FILES = [
    "baryon_density.f32", "temperature.f32", "dark_matter_density.f32",
    "velocity_z.f32", "velocity_x.f32", "velocity_y.f32",
]
NYX_TARGET_STEM = "baryon_density"


# ── Sibling (aux) protocol, identical to sec_4_evaluation/SPERR_fft.py (AUX_MODE='cr_matched') ──
import json
AUX_MODE       = os.environ.get("BO_AUX_MODE", "cr_matched")     # "cr_matched" (paper) | "orig"
AUX_CR_LEVELS  = (100, 200, 300, 400, 500, 600)
AUX_STREAM_DIR = os.path.join(P("ADAMIT_CACHE_DIR"), "aux_streams")


def aux_at_cr_level_sperr(a_path, target_cr):
    """Sibling as the decoder holds it: SPERR-archived at the CR level nearest (log-CR) to
    the target's CR, decompressed with sperr3d from the shared on-disk stream."""
    level = min(AUX_CR_LEVELS, key=lambda L: abs(np.log(L) - np.log(max(target_cr, 1e-9))))
    stem  = os.path.join(AUX_STREAM_DIR, f"sperr_{os.path.basename(a_path).replace('.', '_')}_cr{level}")
    assert os.path.isfile(stem + ".bin"), f"missing {stem}.bin -- run `python sec_4_evaluation/SPERR_fft.py --task aux_prep`"
    rec = f"/tmp/sperr_bo_aux_{os.getpid()}_{time.time_ns()}.dec"
    subprocess.run([SPERR_BIN, "-d", "--decomp_f", rec, stem + ".bin"], capture_output=True, text=True, env=_SPERR_ENV)
    dec = np.ascontiguousarray(np.fromfile(rec, dtype=np.float32).reshape(NYX_SHAPE)); os.remove(rec)
    meta = json.load(open(stem + ".json"))
    print(f"  [aux] {os.path.basename(a_path):24s} target CR {target_cr:6.1f} -> level {level} (sperr): CR {meta['cr']:6.1f}")
    return dec


def load_nyx(target_stem=NYX_TARGET_STEM, rel_err=1e-5):
    """Same operating point as the SZ3 figure: SZ3 at `rel_err` fixes the CR, SPERR is
    bisected to that CR. Returns (Xs, Xps_list, sperr_cr, sperr_bytes, sperr_rel)."""
    fname   = f"{target_stem}.f32"
    gt_path = NYX_BASE + fname
    gt  = np.fromfile(gt_path, dtype=np.float32).reshape(NYX_SHAPE)
    b, sz_cr = sz_engine.compress(gt, 1, 0, float(rel_err), 0); del b
    print(f"  SZ3 rel={rel_err:.0e} -> CR {sz_cr:.1f}; matching SPERR to it ...")
    recon, nb, cr, rel = sperr_at_cr(gt_path, gt, float(sz_cr), label="SPERR/NYX")
    if AUX_MODE == "cr_matched":
        # Sec. 4.1 protocol: siblings archived by SPERR at the CR level nearest the target's
        # CR (streams from `python sec_4_evaluation/SPERR_fft.py --task aux_prep`), same for train/infer.
        aux = [aux_at_cr_level_sperr(NYX_BASE + f, float(cr)) for f in NYX_FIELD_FILES if f != fname]
    else:                                   # "orig": lossless siblings (pre-2026-08-28 runs)
        aux = [np.fromfile(NYX_BASE + f, dtype=np.float32).reshape(NYX_SHAPE)
               for f in NYX_FIELD_FILES if f != fname]
    return [gt] + aux, [recon] + aux, cr, nb, rel


# ── Miranda: single-field, stored 1024×1024×1024 float32 ──────────────────────
MIR_RAW   = P("ADAMIT_MIRANDA_FILE")
MIR_SHAPE = (1024, 1024, 1024)


def load_miranda(rel_err=6.9948e-03):
    vol = np.fromfile(MIR_RAW, dtype=np.float32).reshape(MIR_SHAPE)
    b, sz_cr = sz_engine.compress(vol, 1, 0, float(rel_err), 0); del b
    print(f"  SZ3 rel={rel_err:.0e} -> CR {sz_cr:.1f}; matching SPERR to it ...")
    recon, nb, cr, rel = sperr_at_cr(MIR_RAW, vol, float(sz_cr), label="SPERR/Miranda")
    return [vol], [recon], cr, nb, rel


print("SPERR loaders ready")


SZ engine loaded | sperr3d: True
SPERR loaders ready


In [4]:
# pick_bg_h_under_budget() defaults to h_candidates=range(3, 30), which caps the model
# at h=29. Fine for NYX (30k budget -> h=21) but it silently truncates Miranda: a 240k
# budget needs h=61. Match SPERR_fft.py's range.
H_CANDIDATES = list(range(3, 256))


def run_two_phase(Xs, Xps_list, *, dataset_name, total_time, test_rel_err,
                  tune_depth, freq_warmup, sz_cr=None,
                  n_trials=10, phase1_frac=0.10,
                  param_budget=30000, directions=None,
                  lr_range=(1e-3, float(os.environ.get("BO_LR_MAX", "1e-2"))),
                  proxy_depth_stride=8, proxy_spatial=4, eval_slices=16,
                  min_axis_spread_db=0.30, min_lr_spread_db=0.02,
                  lr_low_reject_frac=0.15):
    """Phase-1 proxy BO + Phase-2 full-resolution sweep, mirroring SPERR_fft.py's
    production path (`_phase1_best_fast` + `_train_residual`) so this figure shows what
    the pipeline actually does:

      * proxy    : per-direction strided subvolume -- stride `proxy_depth_stride` along
                   the direction's OWN axis, then `proxy_spatial` in plane
                   (NYX 512^3 -> 64x128x128, Miranda 1024^3 -> 128x256x256)
      * score    : absolute PSNR of the reconstructed proxy over its middle
                   `eval_slices` slices, used only to RANK trials against one another
      * budget   : Phase 1 gets phase1_frac of total_time (Optuna timeout), Phase 2 the
                   remainder; the cuDNN warm-up trial runs off the clock
      * trust    : two gates on the proxy-PSNR scale, but at different thresholds
                   because they measure quantities of very different size. The three
                   directions stride along different axes and therefore sample
                   different voxels, so their scores differ by whole dB (NYX: 3.3 dB)
                   and 0.30 dB is a sensible bar. Within one direction the voxels are
                   identical and only training differs, so the spread is 0.00-0.09 dB;
                   0.02 dB is ~2.5x the measured run-to-run drift of a trial score
                   (0.008 dB) and is the smallest bar that is still evidence. The direction is
                   adopted only if the best-per-direction scores span more than the
                   gate; the learning rate only if the trials on the chosen direction
                   do. Either dimension falls back to its default when its candidates
                   are not separated by more than the proxy's evaluation noise. Without
                   this, four of five SPERR operating points on NYX baryon density
                   degrade to +0.00 dB (measured: the proxy adopts an interval-endpoint
                   lr, Phase 2 learns nothing in budget, and the error-bounded clamp
                   returns the base reconstruction).
    """
    data_shape = Xs[0].shape
    DS, SP = int(proxy_depth_stride), int(proxy_spatial)
    ENQUEUE_LR = min(max(1e-3, lr_range[0]), lr_range[1])
    if directions is None:
        directions = DIRECTIONS

    PHASE1_TIME   = total_time * phase1_frac
    # 0.6x head-room: the cap governs pure training only; model build and the scoring
    # inference eat the rest of each trial's slot, and all n_trials must fit PHASE1_TIME.
    PER_TRIAL_CAP = (PHASE1_TIME / n_trials) * 0.6

    print(f"\n{'='*64}\n[{dataset_name}] total={total_time:.0f}s  "
          f"Phase1<= {PHASE1_TIME:.1f}s  rel={test_rel_err:.0e}\n{'='*64}")

    # ── Phase-1 proxy: strided along each direction's own axis ────────────────────
    def make_proxy(fields, direction):
        axis, fwd = _DIR_AXIS[direction], _FWD[direction]
        idx = np.arange(0, fields[0].shape[axis], DS)
        return [np.ascontiguousarray(
                    np.transpose(np.take(f, idx, axis=axis), fwd)[:, ::SP, ::SP])
                for f in fields]

    t0 = time.time()
    proxy_cache = {d: (make_proxy(Xs, d), make_proxy(Xps_list, d)) for d in directions}
    proxy_shape = proxy_cache[directions[0]][0][0].shape
    print(f"Proxy cache built in {time.time()-t0:.2f}s | depth stride {DS}, in-plane {SP}")
    for d in directions:
        print(f"  proxy[{d}] shape: {proxy_cache[d][0][0].shape}")

    try:
        bg_h_tune = int(pick_bg_h_under_budget(
            param_budget, shape=proxy_shape, n_fields=len(proxy_cache[directions[0]][1]),
            bg_arch="spatial", h_candidates=H_CANDIDATES)[0])
    except Exception:
        bg_h_tune = 50

    print(f"Phase 1 proxy: {proxy_shape[0]} slices x {proxy_shape[1]}x{proxy_shape[2]} | "
          f"bg_h={bg_h_tune} | {n_trials} trials x {PER_TRIAL_CAP:.2f}s/trial "
          f"| steps/epoch<= {tune_depth} | scored on middle {eval_slices} slices "
          f"| lr_range={lr_range}")

    all_tune_histories = {}
    phase1_start = 0.0   # bound just before study.optimize; objective reads it

    def objective(trial):
        lr        = trial.suggest_float("lr", lr_range[0], lr_range[1], log=True)
        direction = trial.suggest_categorical("direction", directions)
        Xs_sub, Xps_sub = proxy_cache[direction]
        nz       = Xs_sub[0].shape[0]
        steps    = int(min(nz, tune_depth))
        patch_sz = int(min(Xs_sub[0].shape[1], Xs_sub[0].shape[2]))
        evs = int(min(eval_slices, nz))
        z0  = max(0, nz // 2 - evs // 2); z1 = min(nz, z0 + evs)

        t_trial = time.time()
        tune_cfg = build_cfg(
            Xs_sub, Xps_sub, max_train_time=PER_TRIAL_CAP, bg_h=bg_h_tune,
            steps_per_epoch=steps, lr=lr, epochs=999,
            log_prefix=f"BO-{direction}-{lr:.1e}", patch_size=patch_sz)
        tune_cfg.bg_freq_warmup_epochs = freq_warmup
        # Size the lr warmup to the trial, mirroring SPERR_fft.py. A time-capped trial
        # runs ~20-50 steps while bg_stage's default warmup is 200, so the whole trial
        # would sit on the ramp and effectively test lr/9 -- which is why every learning
        # rate on a direction scored identically before this fix.
        tune_cfg.bg_lr_warmup_steps = max(2, int(steps) // 5)

        # Scored on the middle slab only, exactly as the pipeline does: the evaluation
        # must stay cheap relative to the trial's own training time.
        def evaluator(m, cfg=tune_cfg, Xs_d=Xs_sub, Xps_d=Xps_sub, a=z0, b=z1):
            xh = run_bg_inference(unwrap_bg_model(m), Xs_d, Xps_d, cfg, test_rel_err,
                                  z_start=a, z_stop=b)
            return psnr_from_arrays(Xs_d[0][a:b], xh[a:b]), 0.0

        set_seed(42)
        _, hist = train_bg_only(Xs=Xs_sub, Xps=Xps_sub, device=device,
                                cfg=tune_cfg, evaluator=evaluator)
        psnr_vals = [v[1] if isinstance(v, tuple) else v for v in hist.get("psnr", [])]
        final = psnr_vals[-1] if psnr_vals else -1.0
        if not np.isfinite(final):
            final = -1.0          # diverged -> steer BO away instead of aborting the study
        all_tune_histories[(lr, direction)] = hist
        print(f"  Trial {trial.number:2d}: lr={lr:.2e}  dir={direction}  "
              f"PSNR={final:.2f} dB  [{time.time()-t_trial:.2f}s/trial  "
              f"{time.time()-phase1_start:.0f}s elapsed]")
        return final

    # GPU warm-up: pays cuDNN plan creation once, off the Phase-1 clock
    _wXs, _wXps = proxy_cache[directions[0]]
    _wcfg = build_cfg(_wXs, _wXps, max_train_time=0.5, bg_h=bg_h_tune,
                      steps_per_epoch=2, lr=ENQUEUE_LR, epochs=1,
                      log_prefix="warmup", patch_size=_wXs[0].shape[2])
    set_seed(42)
    train_bg_only(Xs=_wXs, Xps=_wXps, device=device, cfg=_wcfg,
                  evaluator=lambda m, cfg=_wcfg: (0.0, 0.0))
    torch.cuda.synchronize() if torch.cuda.is_available() else None
    print("GPU warm-up done (excluded from the Phase-1 budget)")

    # Persist the study so optuna-dashboard can read it:
    #   optuna-dashboard sqlite:///bo_results/optuna_studies.sqlite3   (run from this folder)
    study_name  = f"{dataset_name}_phase1"
    os.makedirs("bo_results", exist_ok=True)
    storage_url = "sqlite:///" + os.path.abspath("bo_results/optuna_studies.sqlite3")
    try:
        optuna.delete_study(study_name=study_name, storage=storage_url)
    except KeyError:
        pass
    study = optuna.create_study(
        direction="maximize", study_name=study_name, storage=storage_url,
        sampler=optuna.samplers.TPESampler(seed=42, n_startup_trials=3))
    for d in directions:
        study.enqueue_trial({"lr": ENQUEUE_LR, "direction": d})

    phase1_start   = time.time()
    # No Optuna timeout: all n_trials MUST run (the per-trial cap already sizes Phase 1
    # to ~PHASE1_TIME; a few-ms overshoot is acceptable, a missing trial is not).
    study.optimize(objective, n_trials=n_trials)
    phase1_elapsed = time.time() - phase1_start

    # ── trust gates (mirror of SPERR_fft.py) ─────────────────────────────────────
    raw_dir = study.best_params["direction"]
    raw_lr  = float(study.best_params["lr"])
    per_dir = {}
    for t in study.trials:
        if t.value is None:
            continue
        d = t.params["direction"]
        if d not in per_dir or t.value > per_dir[d][1]:
            per_dir[d] = (float(t.params["lr"]), float(t.value))
    _axis_best  = [per_dir[d][1] for d in directions if d in per_dir]
    axis_spread = (max(_axis_best) - min(_axis_best)) if len(_axis_best) > 1 else 0.0
    _on_axis    = [t.value for t in study.trials
                   if t.value is not None and t.params["direction"] == raw_dir]
    lr_spread   = (max(_on_axis) - min(_on_axis)) if len(_on_axis) > 1 else 0.0

    if axis_spread <= min_axis_spread_db:
        best_direction = directions[0]
        best_lr = (per_dir[best_direction][0] if best_direction in per_dir
                   else float(ENQUEUE_LR))
        why = (f"axis_spread {axis_spread:.2f}<=tau -> dir {best_direction} (default), "
               f"lr={best_lr:.2e} (best tried on it)")
    elif lr_spread <= min_lr_spread_db:
        best_direction, best_lr = raw_dir, float(ENQUEUE_LR)
        why = (f"dir {best_direction} (axis_spread {axis_spread:.2f}dB), but "
               f"lr_spread {lr_spread:.2f}<=tau -> lr={best_lr:.2e} (default)")
    elif (np.log10(raw_lr) - np.log10(lr_range[0])) <= (
            lr_low_reject_frac * (np.log10(lr_range[1]) - np.log10(lr_range[0]))):
        # A proxy that ranks the SMALLEST lr highest is not recommending a learning rate;
        # it is reporting that it has not trained long enough to tell configurations
        # apart, so the least-perturbed model wins by staying closest to the base. Phase 2
        # gets 85% of the budget, for which "barely train" cannot be the right advice.
        best_direction, best_lr = raw_dir, float(ENQUEUE_LR)
        why = (f"dir {best_direction} (axis_spread {axis_spread:.2f}dB), but raw lr "
               f"{raw_lr:.2e} is in the bottom {lr_low_reject_frac:.0%} of the range "
               f"(proxy still in its damage regime) -> lr={best_lr:.2e} (default)")
    else:
        best_direction, best_lr = raw_dir, raw_lr
        why = (f"dir {best_direction} (axis_spread {axis_spread:.2f}dB), "
               f"lr={best_lr:.2e} (lr_spread {lr_spread:.2f}dB)")
    print(f"  [gates] raw pick lr={raw_lr:.2e} d={raw_dir} | {why}")

    n_done = len(study.trials)
    # Cap the subtraction at the NOMINAL Phase-1 budget: Optuna's timeout only stops
    # NEW trials, so Phase 1 can overshoot slightly, and that overshoot must not be
    # billed to Phase 2 (which the paper reports as 90% of the budget).
    PHASE2_TIME = total_time - min(phase1_elapsed, PHASE1_TIME)
    print(f"\n[{dataset_name}] Phase 1 done in {phase1_elapsed:.1f}s "
          f"(nominal {PHASE1_TIME:.1f}s, {n_done} trials) | Phase 2 budget {PHASE2_TIME:.1f}s")

    # ── Phase 2: re-run every (lr, dir) combo at full resolution ─────────────────
    try:
        bg_h_p2 = int(pick_bg_h_under_budget(
            param_budget, shape=Xs[0].shape, n_fields=len(Xps_list),
            bg_arch="spatial", h_candidates=H_CANDIDATES)[0])
    except Exception:
        bg_h_p2 = 50

    full_histories = {}
    for direction_p2 in directions:
        lrs_this_dir = [lr for (lr, d) in all_tune_histories if d == direction_p2]
        if not lrs_this_dir:
            continue
        t0 = time.time()
        Xs_perm  = permute_fields(Xs, direction_p2)
        Xps_perm = permute_fields(Xps_list, direction_p2)
        n_depth  = Xs_perm[0].shape[0]
        patch_sz = Xs_perm[0].shape[2]
        print(f"\n# [{dataset_name}] dir {direction_p2}: {len(lrs_this_dir)} configs "
              f"(permute {time.time()-t0:.1f}s)")

        for lr_p2 in lrs_this_dir:
            p2_cfg = build_cfg(
                Xs_perm, Xps_perm, max_train_time=PHASE2_TIME, bg_h=bg_h_p2,
                steps_per_epoch=n_depth, lr=lr_p2, epochs=200,
                log_prefix=f"P2-{direction_p2}-{lr_p2:.1e}", patch_size=patch_sz)
            p2_cfg.bg_early_stop         = False   # disabled in all reported experiments
            p2_cfg.bg_freq_warmup_epochs = freq_warmup
            # A wall-clock budget leaves the cosine's T_max effectively infinite, so lr
            # would sit at its peak forever. The pipeline re-fits the schedule to the
            # steps that actually fit the budget once epoch costs are known.
            p2_cfg.bg_sched_time_calibrate = True

            def evaluator_p2(m, cfg=p2_cfg, Xs_p=Xs_perm, Xps_p=Xps_perm,
                             dir_=direction_p2):
                xh_perm = run_bg_inference(unwrap_bg_model(m), Xs_p, Xps_p, cfg, test_rel_err)
                return psnr_from_arrays(Xs[0], unpermute_field(xh_perm, dir_)), 0.0

            set_seed(42)
            _, hist = train_bg_only(Xs=Xs_perm, Xps=Xps_perm, device=device,
                                    cfg=p2_cfg, evaluator=evaluator_p2)
            full_histories[(lr_p2, direction_p2)] = hist
        del Xs_perm, Xps_perm

    def _final(hist):
        ps = [v[1] if isinstance(v, tuple) else v for v in hist.get("psnr", [])]
        ps = [p for p in ps if np.isfinite(p)]   # drop NaN so a diverged run cannot win
        return max(ps) if ps else None           # best-weights, as the pipeline reports
    final_psnr   = {c: _final(h) for c, h in full_histories.items()}
    final_psnr   = {c: v for c, v in final_psnr.items() if v is not None}
    final_winner = max(final_psnr, key=final_psnr.get) if final_psnr else None

    bo_pick = (best_lr, best_direction)
    gap = (final_psnr[final_winner] - final_psnr.get(bo_pick, final_psnr[final_winner])
           if final_winner else float("nan"))
    print(f"\n[{dataset_name}] BO pick {bo_pick} -> "
          f"{final_psnr.get(bo_pick, float('nan')):.2f} dB | "
          f"true best {final_winner} -> {final_psnr.get(final_winner, float('nan')):.2f} dB "
          f"(gap {gap:.2f} dB)")

    return dict(
        dataset_name=dataset_name, total_time=total_time, test_rel_err=test_rel_err,
        tune_depth=tune_depth, data_shape=data_shape, sz_cr=sz_cr,
        study_trials=[(t.number, t.params["lr"], t.params["direction"], t.value)
                      for t in study.trials],
        all_tune_histories=all_tune_histories, full_histories=full_histories,
        best_lr=best_lr, best_direction=best_direction,
        axis_spread=axis_spread, lr_spread=lr_spread,
        final_winner=final_winner, final_psnr=final_psnr,
        phase1_elapsed=phase1_elapsed, phase2_time=PHASE2_TIME,
        per_trial_cap=PER_TRIAL_CAP, n_done=n_done,
    )

print("run_two_phase ready")


run_two_phase ready


In [5]:
# ── NYX: 10 s budget (SPERR base) ──────────────────────────────────────────────────────────
NYX_REL = 1e-5
Xs, Xps_list, sz_cr, sz_bytes, SPERR_REL_NYX = load_nyx(rel_err=NYX_REL)
print(f"NYX loaded | shape {Xs[0].shape} | {len(Xs)} fields | "
      f"SPERR CR={sz_cr:.2f}x (matched to SZ3 rel={NYX_REL:.0e}) rel(max)={SPERR_REL_NYX:.2e}")

result_nyx = run_two_phase(
    Xs, Xps_list, dataset_name="NYX", total_time=10.0, test_rel_err=SPERR_REL_NYX,
    tune_depth=32, freq_warmup=1, sz_cr=sz_cr, param_budget=30000,
    lr_range=(1e-3, float(os.environ.get("BO_LR_MAX", "1e-2"))), proxy_depth_stride=8, proxy_spatial=4, eval_slices=16,
)

# Free the large NYX volumes before loading Miranda
# ── persist a slim copy of the result so the figure can be restyled without retraining ──
import pickle
def _slim(R):
    keep = lambda h: {k: list(h.get(k, [])) for k in ("time", "psnr", "loss") if k in h}
    S = dict(R)
    S["all_tune_histories"] = {k: keep(v) for k, v in R["all_tune_histories"].items()}
    S["full_histories"]     = {k: keep(v) for k, v in R["full_histories"].items()}
    return S
os.makedirs("bo_results", exist_ok=True)
pickle.dump(_slim(result_nyx), open(f"bo_results/{BO_TAG}_nyx{os.environ.get('BO_OUT_SUFFIX', '')}.pkl", "wb"))
print("saved bo_results/" + f"{BO_TAG}_nyx.pkl")

del Xs, Xps_list
import gc; gc.collect()
torch.cuda.empty_cache() if torch.cuda.is_available() else None
print("NYX done; volumes freed")


  SZ3 rel=1e-05 -> CR 439.6; matching SPERR to it ...


    [SPERR/NYX]  1/12 psnr= 110.0 -> CR    62.4 (target 439.6)


    [SPERR/NYX]  2/12 psnr=  65.0 -> CR  1823.7 (target 439.6)


    [SPERR/NYX]  3/12 psnr=  87.5 -> CR   331.3 (target 439.6)


    [SPERR/NYX]  4/12 psnr=  76.2 -> CR   760.9 (target 439.6)


...


[Model: spatial] Total Params: 29,803
 [Params] Main (BG) Network : 29,803 parameters


P2-X-2.3e-03 [Init] Epoch   0 | Global PSNR: 110.67 dB | MaxErr: 0.0
P2-X-2.3e-03 [plan] pure_train_budget=9.00s | epochs_cap=200 | steps/epoch=512 | patch=512 | batch=1 | sample=sequential | data_parallel=False | amp=bf16
P2-X-2.3e-03 [lr-sched] warmup_steps=200 (of 102400 total, cap=20%) -> cosine decay | freq_weight ramps linearly to 1.0 over 1 epoch(s)
P2-X-2.3e-03 [early-stop] DISABLED (cfg.bg_early_stop is False/unset)


P2-X-2.3e-03 [gpu-sampling] 6 fields resident on cuda:0 (~6.4 GB)


P2-X-2.3e-03 Epoch   1 [BG] | train_wall=4.49s | Loss: 1.417411 | Freq: 1.759892 | Global: 112.60 dB | MaxErr: 0.0  [New Best!]
P2-X-2.3e-03 [timing] first_epoch_pure_train≈4.941s (excludes this epoch's end-of-epoch eval)
P2-X-2.3e-03 [lr-sched] time-budget calibration@ep1: epoch=4.49s -> cosine 1.00->0 over ~513 steps (4.5s of 9.0s budget)


P2-X-2.3e-03 Epoch   2 [BG] | train_wall=4.07s | Loss: 1.432168 | Freq: 0.968216 | Global: 114.19 dB | MaxErr: 0.0  [New Best!]

P2-X-2.3e-03 --- Experiment [BG_only] finished ---
P2-X-2.3e-03 --- Pure training time: 9.01 s ---
P2-X-2.3e-03 [timing] epochs=2 | train_wall/epoch: mean=4.28s min=4.07s max=4.49s | sum=8.56s
P2-X-2.3e-03 --- Best global PSNR: 114.19 dB ---

[NYX] BO pick (0.001, 'Y') -> 112.76 dB | true best (0.00959862148464161, 'Y') -> 114.36 dB (gap 1.61 dB)
NYX done; volumes freed


In [6]:
# ── Miranda: 80 s budget (SPERR base) ───────────────────────
MIR_REL = 6.9948e-03
Xs, Xps_list, sz_cr, sz_bytes, SPERR_REL_MIR = load_miranda(rel_err=MIR_REL)
print(f"Miranda loaded | shape {Xs[0].shape} | {len(Xs)} field(s) | "
      f"SPERR CR={sz_cr:.2f}x (matched to SZ3 rel={MIR_REL:.0e}) rel(max)={SPERR_REL_MIR:.2e}")

result_mir = run_two_phase(
    Xs, Xps_list, dataset_name="Miranda", total_time=80.0, test_rel_err=SPERR_REL_MIR,
    tune_depth=64, freq_warmup=1, sz_cr=sz_cr, param_budget=240000,
    lr_range=(1e-3, float(os.environ.get("BO_LR_MAX", "1e-2"))), proxy_depth_stride=8, proxy_spatial=4, eval_slices=16,
)

# ── persist a slim copy of the result so the figure can be restyled without retraining ──
import pickle
def _slim(R):
    keep = lambda h: {k: list(h.get(k, [])) for k in ("time", "psnr", "loss") if k in h}
    S = dict(R)
    S["all_tune_histories"] = {k: keep(v) for k, v in R["all_tune_histories"].items()}
    S["full_histories"]     = {k: keep(v) for k, v in R["full_histories"].items()}
    return S
os.makedirs("bo_results", exist_ok=True)
pickle.dump(_slim(result_mir), open(f"bo_results/{BO_TAG}_mir{os.environ.get('BO_OUT_SUFFIX', '')}.pkl", "wb"))
print("saved bo_results/" + f"{BO_TAG}_mir.pkl")

del Xs, Xps_list
import gc; gc.collect()
torch.cuda.empty_cache() if torch.cuda.is_available() else None
print("Miranda done; volumes freed")


  SZ3 rel=7e-03 -> CR 144.4; matching SPERR to it ...


    [SPERR/Miranda]  1/12 psnr= 110.0 -> CR     6.8 (target 144.4)


    [SPERR/Miranda]  2/12 psnr=  65.0 -> CR    41.8 (target 144.4)


    [SPERR/Miranda]  3/12 psnr=  42.5 -> CR   201.8 (target 144.4)


    [SPERR/Miranda]  4/12 psnr=  53.8 -> CR    86.8 (target 144.4)


...

P2-X-2.1e-03 [gpu-sampling] 1 fields resident on cuda:0 (~8.6 GB)


P2-X-2.1e-03 Epoch   1 [BG] | train_wall=19.65s | Loss: 1.002846 | Freq: 0.845634 | Global: 59.18 dB | MaxErr: 0.0  [New Best!]
P2-X-2.1e-03 [timing] first_epoch_pure_train≈20.259s (excludes this epoch's end-of-epoch eval)
P2-X-2.1e-03 [lr-sched] time-budget calibration@ep1: epoch=19.65s -> cosine 1.00->0 over ~2737 steps (52.5s of 72.2s budget)


P2-X-2.1e-03 Epoch   2 [BG] | train_wall=19.69s | Loss: 1.147336 | Freq: 0.620846 | Global: 60.01 dB | MaxErr: 0.0  [New Best!]
P2-X-2.1e-03 [lr-sched] time-budget calibration@ep2: epoch=19.69s -> cosine 0.69->0 over ~1708 steps (32.8s of 72.2s budget)


P2-X-2.1e-03 Epoch   3 [BG] | train_wall=19.77s | Loss: 1.104908 | Freq: 0.593407 | Global: 60.49 dB | MaxErr: 0.0  [New Best!]


P2-X-2.1e-03 Epoch   4 [BG] | train_wall=12.45s | Loss: 1.024113 | Freq: 0.568151 | Global: 60.62 dB | MaxErr: 0.0  [New Best!]

P2-X-2.1e-03 --- Experiment [BG_only] finished ---
P2-X-2.1e-03 --- Pure training time: 72.19 s ---
P2-X-2.1e-03 [timing] epochs=4 | train_wall/epoch: mean=17.89s min=12.45s max=19.77s | sum=71.56s
P2-X-2.1e-03 --- Best global PSNR: 60.62 dB ---

[Miranda] BO pick (0.0038947060894277247, 'X') -> 60.63 dB | true best (0.003525882465697755, 'Y') -> 60.64 dB (gap 0.01 dB)
Miranda done; volumes freed


In [ ]:
import os, pickle
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import matplotlib.patheffects as pe

# ── Combined 1x4 figure: Phase 1 | Phase 2 for NYX and Miranda ─────────────────
# Style: one colour family per slice direction (X green / Y orange / Z blue), lr
# encoded as shade within the family (light = small lr, dark = large lr); every curve
# additionally gets its own linestyle and hollow marker; ★BO is drawn thickest with a
# large black-edged hollow marker, best is outlined in black.
# Results are loaded from bo_results/<TAG>_{nyx,mir}.pkl when the run cells were not
# executed in this kernel, so the style can be iterated without retraining.
TAG      = BO_TAG                     # "sz3" or "sperr" (set in the run cells)
FIG_SIZE = (48, 11.5)
WSPACE   = 0.24
LEGEND_Y = 0.02
TITLE_FS = LABEL_FS = TICK_FS = 40
LEGEND_FS = LEGEND_TITLE_FS = 38
LW_NORM, LW_BEST, LW_PICK = 4.0, 5.0, 7.5
MS_NORM, MS_PICK = 18, 26
MEW_NORM, MEW_PICK = 3.0, 4.5   # marker edge width (hollow markers)
_LS = ["-", "--", "-.", ":", (0, (5, 1.5)), (0, (3, 1, 1, 1)), (0, (1, 1)), (0, (6, 2, 1, 2))]
_FAMILY  = {"X": plt.cm.Greens, "Y": plt.cm.Oranges, "Z": plt.cm.Blues}
_MARKERS = ["o", "s", "^", "D", "v", "P", "X", "*", "<", ">", "h", "p", "8", "H"]

if "result_nyx" not in globals():
    result_nyx = pickle.load(open(f"bo_results/{TAG}_nyx.pkl", "rb"))
    result_mir = pickle.load(open(f"bo_results/{TAG}_mir.pkl", "rb"))
    print("loaded results from bo_results/ (no retraining)")

def _psnr_list(hist):
    return [v[1] if isinstance(v, tuple) else v for v in hist.get("psnr", [])]

def _fmt_lr(lr):
    return f"{lr:.1e}".replace("e-0", "e-").replace("e+0", "e+")

def _style_map(R):
    """(lr, dir) -> dict(color, marker): shade by lr rank within its direction,
    marker unique per configuration (sorted by direction then lr)."""
    cfgs = sorted(set(R["full_histories"].keys()) | {(lr, d) for (_, lr, d, _) in R["study_trials"]},
                  key=lambda k: (k[1], k[0]))
    style = {}
    for d in ("X", "Y", "Z"):
        lrs = sorted({lr for (lr, dd) in cfgs if dd == d})
        for i, lr in enumerate(lrs):
            shade = 0.45 + 0.5 * (i / max(1, len(lrs) - 1))
            style[(lr, d)] = dict(color=_FAMILY[d](shade))
    for i, k in enumerate(cfgs):
        style[k]["marker"] = _MARKERS[i % len(_MARKERS)]
        style[k]["ls"]     = _LS[i % len(_LS)]
    return style

def _draw(ax, x, y, st, *, is_pick, is_best, label):
    lw = LW_PICK if is_pick else (LW_BEST if is_best else LW_NORM)
    kw = dict(color=st["color"], linewidth=lw, linestyle=st["ls"], marker=st["marker"],
              markersize=MS_PICK if is_pick else MS_NORM,
              markeredgewidth=MEW_PICK if (is_pick or is_best) else MEW_NORM,
              markerfacecolor="white",                       # hollow markers everywhere
              markeredgecolor=("black" if (is_pick or is_best) else st["color"]),
              zorder=10 if is_pick else (8 if is_best else 3), label=label)
    if is_best and not is_pick:
        kw["path_effects"] = [pe.Stroke(linewidth=lw + 3.5, foreground="black"), pe.Normal()]
    ax.plot(x, y, **kw)

def plot_phase1(ax, R, style):
    trials = sorted(R["study_trials"], key=lambda x: x[0])
    t_off = 0.0
    for (num, lr_t, d_t, val) in trials:
        hist = R["all_tune_histories"].get((lr_t, d_t), {})
        t_raw, p_raw = hist.get("time", []), _psnr_list(hist)
        if not t_raw or not p_raw:
            t_off += R["per_trial_cap"]; continue
        t_abs = [t_off + tt for tt in t_raw]
        is_pick = (lr_t == R["best_lr"] and d_t == R["best_direction"])
        _draw(ax, t_abs, p_raw, style[(lr_t, d_t)], is_pick=is_pick, is_best=False,
              label=f"lr={_fmt_lr(lr_t)}, d={d_t}" + (" ★BO" if is_pick else ""))
        t_off = t_abs[-1]
    ax.grid(True, alpha=0.6); ax.tick_params(axis="both", labelsize=TICK_FS)

def plot_phase2(ax, R, style):
    fw, pick = R["final_winner"], (R["best_lr"], R["best_direction"])
    for (lr, d), hist in sorted(R["full_histories"].items(), key=lambda kv: (kv[0][1], kv[0][0])):
        t_vals, p_vals = hist.get("time", []), _psnr_list(hist)
        if not t_vals or not p_vals:
            continue
        is_best, is_pick = (lr, d) == fw, (lr, d) == pick
        suffix = (" ★BO" if is_pick else "") + (" (best)" if is_best else "")
        _draw(ax, t_vals, p_vals, style[(lr, d)], is_pick=is_pick, is_best=is_best,
              label=f"lr={_fmt_lr(lr)}, d={d}{suffix}")
    ax.grid(True, alpha=0.6); ax.tick_params(axis="both", labelsize=TICK_FS)

def _fit_y_decimals(ax, nbins=4):
    ax.yaxis.set_major_locator(ticker.MaxNLocator(nbins=nbins))
    lo, hi = ax.get_ylim(); span = max(hi - lo, 1e-12)
    dec = int(np.clip(np.ceil(-np.log10(span / nbins)) + 1, 1, 4))
    ax.yaxis.set_major_formatter(ticker.FormatStrFormatter(f"%.{dec}f"))

st_nyx, st_mir = _style_map(result_nyx), _style_map(result_mir)
fig, axes = plt.subplots(1, 4, figsize=FIG_SIZE, gridspec_kw={"wspace": WSPACE})
plot_phase1(axes[0], result_nyx, st_nyx); plot_phase2(axes[1], result_nyx, st_nyx)
plot_phase1(axes[2], result_mir, st_mir); plot_phase2(axes[3], result_mir, st_mir)
for ax, t in zip(axes, ["NYX (Phase 1)", "NYX (Phase 2)", "Miranda (Phase 1)", "Miranda (Phase 2)"]):
    ax.set_title(t, fontsize=TITLE_FS, fontweight="bold")
    ax.yaxis.get_major_formatter().set_useOffset(False)
    ax.xaxis.get_major_formatter().set_useOffset(False)
_fit_y_decimals(axes[0]); _fit_y_decimals(axes[2])
axes[1].yaxis.set_major_locator(ticker.MaxNLocator(nbins=5)); axes[1].yaxis.set_major_formatter(ticker.FormatStrFormatter("%.0f"))
axes[3].yaxis.set_major_locator(ticker.MaxNLocator(nbins=5)); axes[3].yaxis.set_major_formatter(ticker.FormatStrFormatter("%.1f"))
fig.supxlabel("Wall Time (s)", fontsize=LABEL_FS, fontweight="bold", y=0.0)
fig.supylabel("PSNR (dB)", fontsize=LABEL_FS, fontweight="bold", x=0.08)

_base = {"sz3": "SZ3", "sperr": "SPERR"}[TAG]
for ax_src, xpos, ttl in ((axes[1], 0.30, f"NYX ({_base} base)"), (axes[3], 0.74, f"Miranda ({_base} base)")):
    h, l = ax_src.get_legend_handles_labels()
    fig.legend(h, l, loc="upper center", bbox_to_anchor=(xpos, LEGEND_Y), ncol=2,
               fontsize=LEGEND_FS, title=ttl, title_fontsize=LEGEND_TITLE_FS,
               handlelength=4.0, markerscale=0.8, columnspacing=1.5)

out_pdf = {"sz3": "NYX_Miranda_1x4.pdf", "sperr": "SPERR_NYX_Miranda_1x4.pdf"}[TAG]
plt.savefig(out_pdf, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {out_pdf}")
